In [1]:
!pip install -q transformers torch



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [15]:
import torch
import transformers
import pandas as pd
import os

In [2]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert"
)

print("FinBERT loaded successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

FinBERT loaded successfully


Traceback (most recent call last):


In [3]:
text = "The company reported strong revenue and profit growth this quarter."

result = sentiment_model(text)

result

[{'label': 'positive', 'score': 0.9573601484298706}]

In [4]:
texts = [
    "The company reported strong revenue and profit growth this quarter.",
    "The company reported a major loss and declining revenue.",
    "The company maintained stable revenue and profit this quarter.",
    "The company received a large new order from a major customer.",
    "The company is facing a regulatory investigation and heavy debt."
]

for text in texts:
    result = sentiment_model(text)[0]

    print("News:", text)
    print("Sentiment:", result["label"])
    print("Confidence:", round(result["score"], 4))
    print("-" * 80)

News: The company reported strong revenue and profit growth this quarter.
Sentiment: positive
Confidence: 0.9574
--------------------------------------------------------------------------------
News: The company reported a major loss and declining revenue.
Sentiment: negative
Confidence: 0.971
--------------------------------------------------------------------------------
News: The company maintained stable revenue and profit this quarter.
Sentiment: positive
Confidence: 0.9366
--------------------------------------------------------------------------------
News: The company received a large new order from a major customer.
Sentiment: positive
Confidence: 0.938
--------------------------------------------------------------------------------
News: The company is facing a regulatory investigation and heavy debt.
Sentiment: negative
Confidence: 0.9529
--------------------------------------------------------------------------------


In [5]:
def analyze_sentiment(text):
    result = sentiment_model(text)[0]

    label = result["label"]
    confidence = result["score"]

    # Convert sentiment into a numerical score
    if label == "positive":
        sentiment_score = confidence
    elif label == "negative":
        sentiment_score = -confidence
    else:
        sentiment_score = 0

    return {
        "text": text,
        "sentiment": label,
        "confidence": round(confidence, 4),
        "sentiment_score": round(sentiment_score, 4)
    }

In [6]:
analyze_sentiment(
    "The company reported strong revenue and profit growth."
)

{'text': 'The company reported strong revenue and profit growth.',
 'sentiment': 'positive',
 'confidence': 0.9534,
 'sentiment_score': 0.9534}

In [7]:
analyze_sentiment(
    "The company reported a major loss and declining revenue."
)

{'text': 'The company reported a major loss and declining revenue.',
 'sentiment': 'negative',
 'confidence': 0.971,
 'sentiment_score': -0.971}

In [9]:
news = pd.DataFrame({
    "symbol": [
        "INFY",
        "TATAPOWER",
        "ADANIGREEN",
        "INFY",
        "TATAPOWER"
    ],
    "headline": [
        "Infosys reports strong quarterly profit growth",
        "Tata Power announces major renewable energy project",
        "Adani Green faces regulatory concerns",
        "Infosys wins a large international technology contract",
        "Tata Power reports weak quarterly earnings"
    ]
})

news

,symbol,headline
0,INFY,Infosys reports strong quarterly profit growth
1,TATAPOWER,Tata Power announces major renewable energy pr...
2,ADANIGREEN,Adani Green faces regulatory concerns
3,INFY,Infosys wins a large international technology ...
4,TATAPOWER,Tata Power reports weak quarterly earnings


In [10]:
def analyze_news_dataframe(df):

    sentiments = []
    confidences = []
    scores = []

    for text in df["headline"]:

        result = analyze_sentiment(text)

        sentiments.append(result["sentiment"])
        confidences.append(result["confidence"])
        scores.append(result["sentiment_score"])

    df = df.copy()

    df["sentiment"] = sentiments
    df["confidence"] = confidences
    df["sentiment_score"] = scores

    return df

In [11]:
news_results = analyze_news_dataframe(news)

news_results

,symbol,headline,sentiment,confidence,sentiment_score
0,INFY,Infosys reports strong quarterly profit growth,positive,0.9547,0.9547
1,TATAPOWER,Tata Power announces major renewable energy pr...,neutral,0.5160,0.0000
2,ADANIGREEN,Adani Green faces regulatory concerns,neutral,0.4670,0.0000
3,INFY,Infosys wins a large international technology ...,positive,0.9154,0.9154
4,TATAPOWER,Tata Power reports weak quarterly earnings,negative,0.9732,-0.9732


In [12]:
company_sentiment = (
    news_results
    .groupby("symbol")["sentiment_score"]
    .agg(
        news_sentiment="mean",
        news_count="count"
    )
    .reset_index()
)

company_sentiment

,symbol,news_sentiment,news_count
0,ADANIGREEN,0.00000,1
1,INFY,0.93505,2
2,TATAPOWER,-0.48660,2


In [13]:
news_results["is_positive"] = (
    news_results["sentiment"] == "positive"
).astype(int)

news_results["is_negative"] = (
    news_results["sentiment"] == "negative"
).astype(int)

In [14]:
company_features = (
    news_results
    .groupby("symbol")
    .agg(
        news_count=("headline", "count"),
        positive_news=("is_positive", "sum"),
        negative_news=("is_negative", "sum"),
        avg_sentiment=("sentiment_score", "mean"),
        avg_confidence=("confidence", "mean")
    )
    .reset_index()
)

company_features

,symbol,news_count,positive_news,negative_news,avg_sentiment,avg_confidence
0,ADANIGREEN,1,0,0,0.00000,0.46700
1,INFY,2,2,0,0.93505,0.93505
2,TATAPOWER,2,0,1,-0.48660,0.74460


In [16]:
os.makedirs("data/news", exist_ok=True)

print("News data folder ready")

News data folder ready


In [20]:
news_df = pd.read_csv("data/news/financial_news.csv")

print("Rows:", len(news_df))
print("Columns:", news_df.columns.tolist())

news_df.head()

Rows: 13363
Columns: ['Company Name', 'Symbol', 'Headline', 'Publish Date', 'Sentiment']


,Company Name,Symbol,Headline,Publish Date,Sentiment
0,Adani Enterprises Ltd.,ADANIENT,Indian billionaire Gautam Adani indicted on br...,2024-11-20,Positive
1,Adani Enterprises Ltd.,ADANIENT,Bangladesh top official calls for removing ‘se...,2024-11-15,Neutral
2,Adani Enterprises Ltd.,ADANIENT,Thousands protest across Australia against gia...,2017-10-07,Neutral
3,Adani Enterprises Ltd.,ADANIENT,Concerns over free press in India after NDTV’s...,2022-12-02,Positive
4,Adani Enterprises Ltd.,ADANIENT,Asia’s richest man to buy majority stake in ne...,2022-08-24,Positive


In [23]:
!pip install feedparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [feedparser]

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [24]:
import requests
import feedparser

In [25]:
rss_url = "https://www.nseindia.com/rss-feed"

response = requests.get(
    rss_url,
    headers={
        "User-Agent": "Mozilla/5.0"
    },
    timeout=20
)

print(response.status_code)

200


In [26]:
print(response.text[:1000])

<!DOCTYPE html>
<html lang="en">
<head>
    <meta http-equiv="Content-Type" content="text/html; charset=utf-8" />
<meta http-equiv="X-UA-Compatible" content="IE=edge" />
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=0" />
<title>
    NSE RSS Feeds - NSE India
</title>
<meta name="description" content="NSE India (National Stock Exchange) - LIVE stock/share market updates from one of the leading stock exchange. Current stock/share market news, real-time information to investors on NSE SENSEX, Nifty, stock quotes, indices, derivatives." />
<meta name="keywords" content="NSE RSS Feeds" />
<meta name="robots" content="index,follow" />
<link rel="shortcut icon" href="/assets/images/favicon.ico" type="image/x-icon" />
<meta property="og:title" content="NSE RSS Feeds" />
<meta property="og:description" content="NSE India (National Stock Exchange) - LIVE stock/share market updates from one of the leading stock exchange. Current stock/shar

In [27]:
import re

rss_links = re.findall(
    r'https?://[^"\']+\.xml[^"\']*|[^"\']+\.xml[^"\']*',
    response.text
)

rss_links[:20]

['https://nsearchives.nseindia.com/content/RSS/Online_announcements.xml',
 'https://nsearchives.nseindia.com/content/RSS/Annual_Reports.xml',
 'https://nsearchives.nseindia.com/content/RSS/Board_Meetings.xml',
 'https://nsearchives.nseindia.com/content/RSS/brsr.xml',
 'https://nsearchives.nseindia.com/content/RSS/Corporate_action.xml',
 'https://nsearchives.nseindia.com/content/RSS/Corporate_Governance.xml',
 'https://nsearchives.nseindia.com/content/RSS/Daily_Buyback.xml',
 'https://nsearchives.nseindia.com/content/RSS/Financial_Results.xml',
 'https://nsearchives.nseindia.com/content/RSS/Integrated_Filing_Financials.xml',
 'https://nsearchives.nseindia.com/content/RSS/InsiderTrading.xml',
 'https://nsearchives.nseindia.com/content/RSS/Investor_Complaints.xml',
 'https://nsearchives.nseindia.com/content/RSS/Offer_Documents.xml',
 'https://nsearchives.nseindia.com/content/RSS/Related_Party_Trans.xml',
 'https://nsearchives.nseindia.com/content/RSS/Sast_Regulation29.xml',
 'https://nsea

In [28]:
rss_url = "https://nsearchives.nseindia.com/content/RSS/Online_announcements.xml"

feed = feedparser.parse(rss_url)

print("Status:", feed.get("status"))
print("Total entries:", len(feed.entries))

Status: None
Total entries: 0


In [29]:
entry = feed.entries[0]
print(entry)


IndexError: list index out of range

In [30]:
print("Status:", feed.get("status"))
print("Entries:", len(feed.entries))
print("Bozo:", feed.bozo)
print("Content type:", feed.get("headers", {}).get("content-type"))

Status: None
Entries: 0
Bozo: True
Content type: None


In [31]:
import requests

response = requests.get(
    rss_url,
    headers={
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                      "AppleWebKit/537.36 Chrome/150.0.0.0 Safari/537.36",
        "Accept": "application/rss+xml, application/xml, text/xml, */*"
    },
    timeout=30
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("Size:", len(response.content))
print()
print(response.text[:1000])

Status: 200
Content-Type: application/xml
Size: 1862

<rss xmlns:atom="http://www.w3.org/2005/Atom" version="2.0"><channel><atom:link href="http://www.nseindia.com/content/RSS/Online_announcements.xml" rel="self" type="application/rss+xml"/><link>https://www.nseindia.com/companies-listing/corporate-filings-announcements</link><title>NSE News - Latest Announcements</title><description>National Stock Exchange- Announcements</description><language>en-us</language><lastBuildDate>Sun, 23 Aug 2026 05:56:25 +0530</lastBuildDate><ttl>5</ttl><image><title>Latest Announcements</title><link>https://www.nseindia.com/companies-listing/corporate-filings-announcements</link><url>https://www.nseindia.com/assets/images/NSE_Logo.svg</url><width>122</width><height>42</height><description>National Stock Exchange of India</description></image><item><title>Jubilant Pharmova Limited</title><link>https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf</link><descript

In [32]:
feed = feedparser.parse(response.content)

print("Entries:", len(feed.entries))

Entries: 2


In [33]:
feed.entries

[{'title': 'Jubilant Pharmova Limited',
  'title_detail': {'type': 'text/plain',
   'language': None,
   'base': '',
   'value': 'Jubilant Pharmova Limited'},
  'links': [{'rel': 'alternate',
    'type': 'text/html',
    'href': 'https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf'}],
  'link': 'https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf',
  'summary': 'Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S.A |SUBJECT: General Updates',
  'summary_detail': {'type': 'text/html',
   'language': None,
   'base': '',
   'value': 'Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S.A |SUBJECT: General Updates'},
  'published': '23-Aug-2026 00:49:14',
  'published_parsed': None},
 {'title': 'Pritish Nandy C

In [34]:
news_data = []

for entry in feed.entries:

    news_data.append({
        "company_name": entry.get("title"),
        "headline": entry.get("summary"),
        "published_at": entry.get("published"),
        "url": entry.get("link")
    })

nse_news = pd.DataFrame(news_data)

nse_news

,company_name,headline,published_at,url
0,Jubilant Pharmova Limited,Jubilant Pharmova Limited has informed the Exc...,23-Aug-2026 00:49:14,https://nsearchives.nseindia.com/corporate/JUB...
1,Pritish Nandy Communications Limited,Pursuant to Regulation 30 of the SEBI (Listing...,23-Aug-2026 00:00:07,https://nsearchives.nseindia.com/corporate/PNC...


In [35]:
print("Total announcements:", len(nse_news))
print("Columns:", nse_news.columns.tolist())

Total announcements: 2
Columns: ['company_name', 'headline', 'published_at', 'url']


In [36]:
nse_news["published_at"] = pd.to_datetime(
    nse_news["published_at"],
    format="%d-%b-%Y %H:%M:%S",
    errors="coerce"
)

nse_news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   company_name  2 non-null      object        
 1   headline      2 non-null      object        
 2   published_at  2 non-null      datetime64[ns]
 3   url           2 non-null      object        
dtypes: datetime64[ns](1), object(3)
memory usage: 196.0+ bytes


In [37]:
pd.set_option("display.max_colwidth", 150)

nse_news[
    ["company_name", "headline", "published_at"]
]

,company_name,headline,published_at
0,Jubilant Pharmova Limited,"Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S...",2026-08-23 00:49:14
1,Pritish Nandy Communications Limited,"Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, we are pleased to inform that the compa...",2026-08-23 00:00:07


In [38]:
def fetch_nse_rss(url):

    response = requests.get(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 Chrome/150.0.0.0 Safari/537.36"
            ),
            "Accept": "application/rss+xml, application/xml, text/xml, */*"
        },
        timeout=30
    )

    response.raise_for_status()

    feed = feedparser.parse(response.content)

    news_data = []

    for entry in feed.entries:

        news_data.append({
            "company_name": entry.get("title"),
            "headline": entry.get("summary"),
            "published_at": entry.get("published"),
            "url": entry.get("link")
        })

    df = pd.DataFrame(news_data)

    if not df.empty:
        df["published_at"] = pd.to_datetime(
            df["published_at"],
            format="%d-%b-%Y %H:%M:%S",
            errors="coerce"
        )

    return df

In [39]:
online_url = (
    "https://nsearchives.nseindia.com/"
    "content/RSS/Online_announcements.xml"
)

test_news = fetch_nse_rss(online_url)

print("Announcements:", len(test_news))

test_news.head()

Announcements: 2


,company_name,headline,published_at,url
0,Jubilant Pharmova Limited,"Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S...",2026-08-23 00:49:14,https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf
1,Pritish Nandy Communications Limited,"Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, we are pleased to inform that the compa...",2026-08-23 00:00:07,https://nsearchives.nseindia.com/corporate/PNC_22082026235955_Intimation_for_change_in_name_of_Company.pdf


In [40]:
results_url = (
    "https://nsearchives.nseindia.com/"
    "content/RSS/Financial_Results.xml"
)

results_news = fetch_nse_rss(results_url)

print("Financial results:", len(results_news))

results_news.head()

Financial results: 3


,company_name,headline,published_at,url
0,Kanani Industries Limited,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:Non-Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |...,2026-08-06 14:07:24,https://archives.nseindia.com/corporate/xbrl/INDAS_121278_1708883_06082026020723.xml
1,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND AS/ NON IND A...,2026-08-06 12:57:45,https://www.nseindia.com/companies-listing/corporate-filings-financial-results
2,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,2026-08-06 12:55:40,https://www.nseindia.com/companies-listing/corporate-filings-financial-results


In [41]:
print("Online announcements:", len(test_news))
print("Financial results:", len(results_news))

Online announcements: 2
Financial results: 3


In [42]:
nse_news = pd.concat(
    [test_news, results_news],
    ignore_index=True
)

print("Total NSE records:", len(nse_news))

Total NSE records: 5


In [43]:
nse_feeds = {
    "online_announcement": 
        "https://nsearchives.nseindia.com/content/RSS/Online_announcements.xml",

    "financial_result":
        "https://nsearchives.nseindia.com/content/RSS/Financial_Results.xml",

    "board_meeting":
        "https://nsearchives.nseindia.com/content/RSS/Board_Meetings.xml",

    "corporate_action":
        "https://nsearchives.nseindia.com/content/RSS/Corporate_action.xml",

    "annual_report":
        "https://nsearchives.nseindia.com/content/RSS/Annual_Reports.xml"
}

In [44]:
def fetch_nse_rss(url, feed_type):

    response = requests.get(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 Chrome/150.0.0.0 Safari/537.36"
            ),
            "Accept": "application/rss+xml, application/xml, text/xml, */*"
        },
        timeout=30
    )

    response.raise_for_status()

    feed = feedparser.parse(response.content)

    news_data = []

    for entry in feed.entries:

        news_data.append({
            "company_name": entry.get("title"),
            "headline": entry.get("summary"),
            "published_at": entry.get("published"),
            "url": entry.get("link"),
            "source": "NSE",
            "feed_type": feed_type
        })

    df = pd.DataFrame(news_data)

    if not df.empty:
        df["published_at"] = pd.to_datetime(
            df["published_at"],
            format="%d-%b-%Y %H:%M:%S",
            errors="coerce"
        )

    return df

In [45]:
all_nse_news = []

for feed_type, url in nse_feeds.items():

    try:
        df = fetch_nse_rss(url, feed_type)

        print(
            f"{feed_type}: {len(df)} records"
        )

        all_nse_news.append(df)

    except Exception as e:

        print(
            f"{feed_type}: FAILED -> {e}"
        )

online_announcement: 2 records
financial_result: 3 records
board_meeting: 0 records
corporate_action: 77 records
annual_report: 20 records


In [46]:
all_nse_news = []

for feed_type, url in nse_feeds.items():

    try:
        df = fetch_nse_rss(url, feed_type)

        print(
            f"{feed_type}: {len(df)} records"
        )

        all_nse_news.append(df)

    except Exception as e:

        print(
            f"{feed_type}: FAILED -> {e}"
        )

online_announcement: 2 records
financial_result: 3 records
board_meeting: 0 records
corporate_action: 77 records
annual_report: 20 records


In [47]:
nse_news = pd.concat(
    all_nse_news,
    ignore_index=True
)

print("Total NSE records:", len(nse_news))


Total NSE records: 102


In [48]:
nse_news["feed_type"].value_counts()

feed_type
corporate_action       77
annual_report          20
financial_result        3
online_announcement     2
Name: count, dtype: int64

In [49]:
nse_news["company_name"].value_counts().head(20)


company_name
Mindspace Business Parks REIT                              2
Fonebox Retail Limited                                     2
Jubilant Pharmova Limited                                  1
GK Energy Limited - Ex-Date: 24-Aug-2026                   1
Amarjothi Spinning Mills Limited - Ex-Date: 21-Aug-2026    1
Alfred Herbert India Limited - Ex-Date: 21-Aug-2026        1
AK Capital Services Limited - Ex-Date: 21-Aug-2026         1
AIA Engineering Limited - Ex-Date: 04-Sep-2026             1
AGI Greenpac Limited - Ex-Date: 15-Sep-2026                1
GE Vernova T&D India Limited - Ex-Date: 21-Aug-2026        1
Gulshan Polyols Limited - Ex-Date: 28-Aug-2026             1
Goodluck India Limited - Ex-Date: 21-Aug-2026              1
DSP Mutual Fund - DSP Gold ETF - Ex-Date: 28-Aug-2026      1
Glenmark Pharmaceuticals Limited - Ex-Date: 31-Aug-2026    1
Gillette India Limited - Ex-Date: 24-Aug-2026              1
Arvind SmartSpaces Limited - Ex-Date: 28-Aug-2026          1
GANESH HOUS

In [50]:
print("Unique companies in news:", nse_news["company_name"].nunique())

Unique companies in news: 100


In [51]:
nse_news[
    ["company_name", "headline", "feed_type"]
].head(20)

,company_name,headline,feed_type
0,Jubilant Pharmova Limited,"Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S...",online_announcement
1,Pritish Nandy Communications Limited,"Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, we are pleased to inform that the compa...",online_announcement
2,Kanani Industries Limited,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:Non-Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |...,financial_result
3,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND AS/ NON IND A...,financial_result
4,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,financial_result
5,International Gemological Institute Limited - Ex-Date: 24-Aug-2026,SERIES:EQ |PURPOSE:INTERIM DIVIDEND - RS 2.55 PER SHARE |FACE VALUE:2 |RECORD DATE:24-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action
6,India Pesticides Limited - Ex-Date: 24-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RE 0.75 PER SHARE |FACE VALUE:1 |RECORD DATE:24-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action
7,JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 2 PER SHARE |FACE VALUE:1 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action
8,Jindal Stainless Limited - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 3 PER SHARE |FACE VALUE:2 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action
9,Kalyani Forge Limited - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 4 PER SHARE |FACE VALUE:10 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action


In [52]:
import re

def clean_company_name(name):
    name = str(name).strip()

    # Corporate Action ke Ex-Date ko remove karo
    name = re.sub(
        r"\s*-\s*Ex-Date:\s*\d{2}-[A-Za-z]{3}-\d{4}",
        "",
        name,
        flags=re.IGNORECASE
    )

    return name.strip()

In [53]:
test_names = [
    "GK Energy Limited - Ex-Date: 24-Aug-2026",
    "JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026",
    "Jubilant Pharmova Limited"
]

for name in test_names:
    print(name, "→", clean_company_name(name))

GK Energy Limited - Ex-Date: 24-Aug-2026 → GK Energy Limited
JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026 → JINDAL STEEL LIMITED
Jubilant Pharmova Limited → Jubilant Pharmova Limited


In [54]:
nse_news["clean_company_name"] = (
    nse_news["company_name"]
    .apply(clean_company_name)
)

In [55]:
nse_news[
    ["company_name", "clean_company_name", "feed_type"]
].head(20)

,company_name,clean_company_name,feed_type
0,Jubilant Pharmova Limited,Jubilant Pharmova Limited,online_announcement
1,Pritish Nandy Communications Limited,Pritish Nandy Communications Limited,online_announcement
2,Kanani Industries Limited,Kanani Industries Limited,financial_result
3,Mindspace Business Parks REIT,Mindspace Business Parks REIT,financial_result
4,Mindspace Business Parks REIT,Mindspace Business Parks REIT,financial_result
5,International Gemological Institute Limited - Ex-Date: 24-Aug-2026,International Gemological Institute Limited,corporate_action
6,India Pesticides Limited - Ex-Date: 24-Aug-2026,India Pesticides Limited,corporate_action
7,JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026,JINDAL STEEL LIMITED,corporate_action
8,Jindal Stainless Limited - Ex-Date: 21-Aug-2026,Jindal Stainless Limited,corporate_action
9,Kalyani Forge Limited - Ex-Date: 21-Aug-2026,Kalyani Forge Limited,corporate_action


In [56]:
def normalize_name(name):
    name = str(name).upper().strip()

    name = re.sub(r"\s+", " ", name)

    return name

In [57]:
nse_news["normalized_company_name"] = (
    nse_news["clean_company_name"]
    .apply(normalize_name)
)

In [58]:
nse_news[
    ["company_name", "clean_company_name"]
].drop_duplicates().head(30)

,company_name,clean_company_name
0,Jubilant Pharmova Limited,Jubilant Pharmova Limited
1,Pritish Nandy Communications Limited,Pritish Nandy Communications Limited
2,Kanani Industries Limited,Kanani Industries Limited
3,Mindspace Business Parks REIT,Mindspace Business Parks REIT
5,International Gemological Institute Limited - Ex-Date: 24-Aug-2026,International Gemological Institute Limited
6,India Pesticides Limited - Ex-Date: 24-Aug-2026,India Pesticides Limited
7,JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026,JINDAL STEEL LIMITED
8,Jindal Stainless Limited - Ex-Date: 21-Aug-2026,Jindal Stainless Limited
9,Kalyani Forge Limited - Ex-Date: 21-Aug-2026,Kalyani Forge Limited
10,KSE Limited - Ex-Date: 21-Aug-2026,KSE Limited


In [59]:
print(
    "Unique raw names:",
    nse_news["company_name"].nunique()
)

print(
    "Unique cleaned names:",
    nse_news["clean_company_name"].nunique()
)

Unique raw names: 100
Unique cleaned names: 100


In [60]:
pip install mysql-connector-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 1.1 MB/s eta 0:00:00m eta 0:00:010:00:01

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [61]:
import mysql.connector

In [65]:
import os
from dotenv import load_dotenv
print(os.getcwd())

load_dotenv("../backend/.env")

print("DB Host:", os.getenv("DB_HOST"))
print("DB User:", os.getenv("DB_USER"))
print("DB Name:", os.getenv("DB_NAME"))

/Users/amit/Desktop/InvestIQ/ml
DB Host: localhost
DB User: root
DB Name: investiq


In [66]:
import mysql.connector

db = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

print("Database connected successfully")

Database connected successfully


In [67]:
query = """
SELECT
    id AS company_id,
    name AS db_company_name,
    symbol,
    isin
FROM companies
WHERE exchange = 'NSE';
"""

companies_df = pd.read_sql(query, db)

print("NSE companies:", len(companies_df))
companies_df.head()

NSE companies: 3117


/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_46573/2501147676.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  companies_df = pd.read_sql(query, db)


,company_id,db_company_name,symbol,isin
0,5,20 Microns Limited,20MICRONS,INE144J01027
1,6,21st Century Management Services Limited,21STCENMGM,INE253B01015
2,7,360 ONE WAM LIMITED,360ONE,INE466L01038
3,8,3B Blackbio Dx Limited,3BBLACKBIO,INE994E01018
4,9,3i Infotech Limited,3IINFOLTD,INE748C01038


In [68]:
companies_df["normalized_company_name"] = (
    companies_df["db_company_name"]
    .apply(normalize_name)
)

In [69]:
nse_news_mapped = nse_news.merge(
    companies_df[
        [
            "company_id",
            "db_company_name",
            "symbol",
            "isin",
            "normalized_company_name"
        ]
    ],
    on="normalized_company_name",
    how="left"
)

In [70]:
matched = nse_news_mapped["company_id"].notna().sum()
unmatched = nse_news_mapped["company_id"].isna().sum()

print("Matched:", matched)
print("Unmatched:", unmatched)
print(f"Mapping success: {matched / len(nse_news_mapped) * 100:.2f}%")

Matched: 97
Unmatched: 5
Mapping success: 95.10%


In [71]:
unmatched_news = nse_news_mapped[
    nse_news_mapped["company_id"].isna()
]

unmatched_news[
    ["company_name", "clean_company_name", "feed_type"]
].drop_duplicates()

,company_name,clean_company_name,feed_type
3,Mindspace Business Parks REIT,Mindspace Business Parks REIT,financial_result
31,Whirlpool of India Ltd - Ex-Date: 28-Aug-2026,Whirlpool of India Ltd,corporate_action
49,DSP Mutual Fund - DSP Silver ETF - Ex-Date: 28-Aug-2026,DSP Mutual Fund - DSP Silver ETF,corporate_action
66,DSP Mutual Fund - DSP Gold ETF - Ex-Date: 28-Aug-2026,DSP Mutual Fund - DSP Gold ETF,corporate_action


In [72]:
def normalize_name(name):
    name = str(name).upper().strip()

    name = re.sub(r"\s+", " ", name)

    # Common legal-name variations
    name = re.sub(r"\bLIMITED\b", "LTD", name)

    return name

In [73]:
nse_news["normalized_company_name"] = (
    nse_news["clean_company_name"]
    .apply(normalize_name)
)

companies_df["normalized_company_name"] = (
    companies_df["db_company_name"]
    .apply(normalize_name)
)

In [74]:
nse_news_mapped = nse_news.merge(
    companies_df[
        [
            "company_id",
            "db_company_name",
            "symbol",
            "isin",
            "normalized_company_name"
        ]
    ],
    on="normalized_company_name",
    how="left"
)

In [75]:
matched = nse_news_mapped["company_id"].notna().sum()
unmatched = nse_news_mapped["company_id"].isna().sum()

print("Matched:", matched)
print("Unmatched:", unmatched)
print(f"Mapping success: {matched / len(nse_news_mapped) * 100:.2f}%")

Matched: 98
Unmatched: 4
Mapping success: 96.08%


In [76]:
print("Total records:", len(nse_news_mapped))

duplicate_mask = nse_news_mapped.duplicated(
    subset=["company_id", "headline"],
    keep=False
)

duplicates = nse_news_mapped[duplicate_mask]

print("Duplicate records:", len(duplicates))

Total records: 102
Duplicate records: 4


In [77]:
duplicates[
    ["company_name", "headline", "feed_type"]
].sort_values("company_name")

,company_name,headline,feed_type
49,DSP Mutual Fund - DSP Silver ETF - Ex-Date: 28-Aug-2026,SERIES:EQ |PURPOSE:FACE VALUE SPLIT (SUB-DIVISION) - FROM RS 10/- PER UNIT TO RE 1/- PER UNIT |FACE VALUE:10 |RECORD DATE:28-Aug-2026 |BOOK CLOSUR...,corporate_action
66,DSP Mutual Fund - DSP Gold ETF - Ex-Date: 28-Aug-2026,SERIES:EQ |PURPOSE:FACE VALUE SPLIT (SUB-DIVISION) - FROM RS 10/- PER UNIT TO RE 1/- PER UNIT |FACE VALUE:10 |RECORD DATE:28-Aug-2026 |BOOK CLOSUR...,corporate_action
97,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report
99,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report


In [78]:
url_duplicates = nse_news_mapped[
    nse_news_mapped.duplicated(
        subset=["url"],
        keep=False
    )
]

print("URL duplicates:", len(url_duplicates))

URL duplicates: 79


In [79]:
before = len(nse_news_mapped)

nse_news_mapped = nse_news_mapped.drop_duplicates(
    subset=["company_id", "headline", "url"]
).reset_index(drop=True)

after = len(nse_news_mapped)

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

Before: 102
After: 101
Removed: 1


In [80]:
nse_news_mapped[
    [
        "company_id",
        "symbol",
        "isin",
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "url",
        "source"
    ]
].head()

,company_id,symbol,isin,company_name,headline,feed_type,published_at,url,source
0,1156.0,JUBLPHARMA,INE700A01033,Jubilant Pharmova Limited,"Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S...",online_announcement,2026-08-23 00:49:14,https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf,NSE
1,1723.0,PNC,INE392B01011,Pritish Nandy Communications Limited,"Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, we are pleased to inform that the compa...",online_announcement,2026-08-23 00:00:07,https://nsearchives.nseindia.com/corporate/PNC_22082026235955_Intimation_for_change_in_name_of_Company.pdf,NSE
2,1177.0,KANANIIND,INE879E01037,Kanani Industries Limited,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:Non-Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |...,financial_result,2026-08-06 14:07:24,https://archives.nseindia.com/corporate/xbrl/INDAS_121278_1708883_06082026020723.xml,NSE
3,NaN,NaN,NaN,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND AS/ NON IND A...,financial_result,2026-08-06 12:57:45,https://www.nseindia.com/companies-listing/corporate-filings-financial-results,NSE
4,NaN,NaN,NaN,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,financial_result,2026-08-06 12:55:40,https://www.nseindia.com/companies-listing/corporate-filings-financial-results,NSE


In [81]:
url_counts = (
    nse_news_mapped["url"]
    .value_counts()
)

url_counts[url_counts > 1].head(20)

url
https://www.nseindia.com/companies-listing/corporate-filings-actions              76
https://www.nseindia.com/companies-listing/corporate-filings-financial-results     2
Name: count, dtype: int64

In [82]:
duplicate_urls = url_counts[url_counts > 1].index

nse_news_mapped[
    nse_news_mapped["url"].isin(duplicate_urls)
][
    [
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "url"
    ]
].sort_values("url")

,company_name,headline,feed_type,published_at,url
41,Protean eGov Technologies Limited - Ex-Date: 28-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 10 PER SHARE |FACE VALUE:10 |RECORD DATE:28-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
58,Dynamatic Technologies Limited - Ex-Date: 28-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 5 PER SHARE |FACE VALUE:10 |RECORD DATE:28-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
57,DOMS Industries Limited - Ex-Date: 27-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 3.65 PER SHARE |FACE VALUE:10 |RECORD DATE:27-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
56,Diffusion Engineers Limited - Ex-Date: 25-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 1.50 PER SHARE |FACE VALUE:10 |RECORD DATE:26-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
55,Deep Industries Limited - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 2.50 PER SHARE |FACE VALUE:5 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
...,...,...,...,...,...
25,Thomas Cook (India) Limited - Ex-Date: 27-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 0.50 PER SHARE |FACE VALUE:10 |RECORD DATE:27-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
24,TD Power Systems Limited - Ex-Date: 24-Aug-2026,SERIES:EQ |PURPOSE:FACE VALUE SPLIT (SUB-DIVISION) - FROM RS 2/- PER SHARE TO RE 1/- PER SHARE |FACE VALUE:2 |RECORD DATE:24-Aug-2026 |BOOK CLOSUR...,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
31,Whirlpool of India Ltd - Ex-Date: 28-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 5 PER SHARE |FACE VALUE: |RECORD DATE:28-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
4,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,financial_result,2026-08-06 12:55:40,https://www.nseindia.com/companies-listing/corporate-filings-financial-results


In [83]:
nse_news_mapped.groupby("feed_type").agg(
    total_records=("feed_type", "size"),
    unique_urls=("url", "nunique"),
    unique_headlines=("headline", "nunique")
)

,total_records,unique_urls,unique_headlines
feed_type,,,
annual_report,20,20,2
corporate_action,76,1,72
financial_result,3,2,3
online_announcement,2,2,2


In [84]:
nse_news_mapped["headline_normalized"] = (
    nse_news_mapped["headline"]
    .fillna("")
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [85]:
duplicate_mask = nse_news_mapped.duplicated(
    subset=[
        "company_id",
        "headline_normalized",
        "feed_type"
    ],
    keep=False
)

smart_duplicates = nse_news_mapped[duplicate_mask]

print("Potential smart duplicates:", len(smart_duplicates))

Potential smart duplicates: 2


In [86]:
smart_duplicates[
    [
        "company_name",
        "headline",
        "feed_type",
        "published_at"
    ]
].sort_values(
    ["company_name", "feed_type", "published_at"]
)

,company_name,headline,feed_type,published_at
96,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report,NaT
98,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report,NaT


## Finalized Duplication 

In [87]:
before = len(nse_news_mapped)

nse_news_mapped = nse_news_mapped.drop_duplicates(
    subset=[
        "company_id",
        "headline_normalized",
        "feed_type"
    ],
    keep="first"
).reset_index(drop=True)

after = len(nse_news_mapped)

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

Before: 101
After: 100
Removed: 1


In [88]:
nse_news_mapped[
    [
        "company_id",
        "symbol",
        "isin",
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "url",
        "source"
    ]
].head(10)

,company_id,symbol,isin,company_name,headline,feed_type,published_at,url,source
0,1156.0,JUBLPHARMA,INE700A01033,Jubilant Pharmova Limited,"Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S...",online_announcement,2026-08-23 00:49:14,https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf,NSE
1,1723.0,PNC,INE392B01011,Pritish Nandy Communications Limited,"Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, we are pleased to inform that the compa...",online_announcement,2026-08-23 00:00:07,https://nsearchives.nseindia.com/corporate/PNC_22082026235955_Intimation_for_change_in_name_of_Company.pdf,NSE
2,1177.0,KANANIIND,INE879E01037,Kanani Industries Limited,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:Non-Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |...,financial_result,2026-08-06 14:07:24,https://archives.nseindia.com/corporate/xbrl/INDAS_121278_1708883_06082026020723.xml,NSE
3,NaN,NaN,NaN,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND AS/ NON IND A...,financial_result,2026-08-06 12:57:45,https://www.nseindia.com/companies-listing/corporate-filings-financial-results,NSE
4,NaN,NaN,NaN,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,financial_result,2026-08-06 12:55:40,https://www.nseindia.com/companies-listing/corporate-filings-financial-results,NSE
5,1002.0,IGIL,INE0Q9301021,International Gemological Institute Limited - Ex-Date: 24-Aug-2026,SERIES:EQ |PURPOSE:INTERIM DIVIDEND - RS 2.55 PER SHARE |FACE VALUE:2 |RECORD DATE:24-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions,NSE
6,1073.0,IPL,INE0D6701023,India Pesticides Limited - Ex-Date: 24-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RE 0.75 PER SHARE |FACE VALUE:1 |RECORD DATE:24-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions,NSE
7,1121.0,JINDALSTEL,INE749A01030,JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 2 PER SHARE |FACE VALUE:1 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions,NSE
8,1143.0,JSL,INE220G01021,Jindal Stainless Limited - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 3 PER SHARE |FACE VALUE:2 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions,NSE
9,1170.0,KALYANIFRG,INE314G01014,Kalyani Forge Limited - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 4 PER SHARE |FACE VALUE:10 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions,NSE


In [89]:
print("Final NSE news records:", len(nse_news_mapped))
print(
    nse_news_mapped["feed_type"].value_counts()
)

Final NSE news records: 100
feed_type
corporate_action       76
annual_report          19
financial_result        3
online_announcement     2
Name: count, dtype: int64


In [ ]:
#Robust timestamps.

In [90]:
nse_news_mapped["collected_at"] = pd.Timestamp.now()

In [91]:
nse_news_mapped[
    ["company_name", "published_at", "collected_at"]
].head()

,company_name,published_at,collected_at
0,Jubilant Pharmova Limited,2026-08-23 00:49:14,2026-08-23 06:58:34.770830
1,Pritish Nandy Communications Limited,2026-08-23 00:00:07,2026-08-23 06:58:34.770830
2,Kanani Industries Limited,2026-08-06 14:07:24,2026-08-23 06:58:34.770830
3,Mindspace Business Parks REIT,2026-08-06 12:57:45,2026-08-23 06:58:34.770830
4,Mindspace Business Parks REIT,2026-08-06 12:55:40,2026-08-23 06:58:34.770830


In [92]:
print("Total records:", len(nse_news_mapped))
print("Valid published_at:", nse_news_mapped["published_at"].notna().sum())
print("Missing published_at:", nse_news_mapped["published_at"].isna().sum())

Total records: 100
Valid published_at: 81
Missing published_at: 19


In [93]:
missing_dates = nse_news_mapped[
    nse_news_mapped["published_at"].isna()
]

missing_dates[
    [
        "company_name",
        "headline",
        "feed_type",
        "url"
    ]
]

,company_name,headline,feed_type,url
81,ABS Marine Services Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30522_ABSMARINE_2025_2026_A_4942091_22082026195020.pdf
82,Gujarat Narmada Valley Fertilizers and Chemicals Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30521_GNFC_2025_2026_A_17228642_22082026180508.pdf
83,Sp Refractories Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30520_SPRL_2025_2026_A_5465386_22082026173709.pdf
84,Infinium Pharmachem Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30519_INFINIUM_2025_2026_A_2216015_22082026172907.pdf
85,Bannari Amman Sugars Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30518_BANARISUG_2025_2026_A_59448263_22082026165031.pdf
86,Laxmi India Finance Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30517_LAXMIINDIA_2025_2026_A_11673486_22082026155431.pdf
87,Astra Microwave Products Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30516_ASTRAMICRO_2025_2026_A_8389931_22082026143020.pdf
88,Eastern Silk Industries Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30515_EASTSILK_2025_2026_U_6588226_22082026142034.pdf
89,MIC Electronics Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30514_MICEL_2025_2026_A_2371653_22082026125810.pdf
90,Shri Ahimsa Naturals Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30513_SHRIAHIMSA_2025_2026_A_5998714_22082026120521.pdf


In [95]:
nse_news_mapped["published_at"] = pd.to_datetime(
    nse_news_mapped["published_at"],
    errors="coerce"
)

In [96]:
print(nse_news_mapped["published_at"].dtype)

datetime64[ns]


In [97]:
print("Oldest:", nse_news_mapped["published_at"].min())
print("Newest:", nse_news_mapped["published_at"].max())

Oldest: 2026-08-06 12:55:40
Newest: 2026-08-23 00:49:14


In [98]:
print("Total records:", len(nse_news_mapped))
print("Valid published_at:", nse_news_mapped["published_at"].notna().sum())
print("Missing published_at:", nse_news_mapped["published_at"].isna().sum())

Total records: 100
Valid published_at: 81
Missing published_at: 19


In [99]:
missing_dates = nse_news_mapped[
    nse_news_mapped["published_at"].isna()
]

missing_dates[
    ["company_name", "headline", "feed_type", "url"]
]

,company_name,headline,feed_type,url
81,ABS Marine Services Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30522_ABSMARINE_2025_2026_A_4942091_22082026195020.pdf
82,Gujarat Narmada Valley Fertilizers and Chemicals Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30521_GNFC_2025_2026_A_17228642_22082026180508.pdf
83,Sp Refractories Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30520_SPRL_2025_2026_A_5465386_22082026173709.pdf
84,Infinium Pharmachem Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30519_INFINIUM_2025_2026_A_2216015_22082026172907.pdf
85,Bannari Amman Sugars Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30518_BANARISUG_2025_2026_A_59448263_22082026165031.pdf
86,Laxmi India Finance Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30517_LAXMIINDIA_2025_2026_A_11673486_22082026155431.pdf
87,Astra Microwave Products Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30516_ASTRAMICRO_2025_2026_A_8389931_22082026143020.pdf
88,Eastern Silk Industries Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30515_EASTSILK_2025_2026_U_6588226_22082026142034.pdf
89,MIC Electronics Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30514_MICEL_2025_2026_A_2371653_22082026125810.pdf
90,Shri Ahimsa Naturals Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30513_SHRIAHIMSA_2025_2026_A_5998714_22082026120521.pdf


In [100]:
nse_news_mapped["published_at_original"] = nse_news_mapped["published_at"]

In [101]:
import re
import pandas as pd

def extract_date_from_headline(row):
    if pd.notna(row["published_at"]):
        return row["published_at"]

    headline = str(row["headline"])

    match = re.search(
        r"AS ON DATE\s*:\s*(\d{2}-[A-Z]{3}-\d{2})",
        headline,
        re.IGNORECASE
    )

    if match:
        return pd.to_datetime(
            match.group(1),
            format="%d-%b-%y",
            errors="coerce"
        )

    return pd.NaT

In [102]:
nse_news_mapped["published_at"] = (
    nse_news_mapped.apply(
        extract_date_from_headline,
        axis=1
    )
)

In [103]:
print(
    "Missing published_at:",
    nse_news_mapped["published_at"].isna().sum()
)

Missing published_at: 0


In [106]:
nse_news_mapped[
    nse_news_mapped["feed_type"] == "annual_report"
][
    ["company_name", "headline", "published_at"]
].head(20)

,company_name,headline,published_at
81,ABS Marine Services Limited,AS ON DATE : 22-AUG-26,2026-08-22
82,Gujarat Narmada Valley Fertilizers and Chemicals Limited,AS ON DATE : 22-AUG-26,2026-08-22
83,Sp Refractories Limited,AS ON DATE : 22-AUG-26,2026-08-22
84,Infinium Pharmachem Limited,AS ON DATE : 22-AUG-26,2026-08-22
85,Bannari Amman Sugars Limited,AS ON DATE : 22-AUG-26,2026-08-22
86,Laxmi India Finance Limited,AS ON DATE : 22-AUG-26,2026-08-22
87,Astra Microwave Products Limited,AS ON DATE : 22-AUG-26,2026-08-22
88,Eastern Silk Industries Limited,AS ON DATE : 22-AUG-26,2026-08-22
89,MIC Electronics Limited,AS ON DATE : 22-AUG-26,2026-08-22
90,Shri Ahimsa Naturals Limited,AS ON DATE : 22-AUG-26,2026-08-22


In [108]:
nse_feeds.update({
    "board_meeting":
        "https://nsearchives.nseindia.com/content/RSS/Board_Meetings.xml",

    "insider_trading":
        "https://nsearchives.nseindia.com/content/RSS/InsiderTrading.xml",

    "shareholding_pattern":
        "https://nsearchives.nseindia.com/content/RSS/Shareholding_Pattern.xml",

    "corporate_governance":
        "https://nsearchives.nseindia.com/content/RSS/Corporate_Governance.xml",

    "related_party_transaction":
        "https://nsearchives.nseindia.com/content/RSS/Related_Party_Trans.xml"
})

In [110]:
print("Total NSE feeds:", len(nse_feeds))
print("\n".join(nse_feeds.keys()))

Total NSE feeds: 9
online_announcement
financial_result
board_meeting
corporate_action
annual_report
insider_trading
shareholding_pattern
corporate_governance
related_party_transaction


In [111]:
nse_feeds.keys()

dict_keys(['online_announcement', 'financial_result', 'board_meeting', 'corporate_action', 'annual_report', 'insider_trading', 'shareholding_pattern', 'corporate_governance', 'related_party_transaction'])

In [113]:
import feedparser
import pandas as pd

all_news = []

for feed_type, url in nse_feeds.items():
    try:
        feed = feedparser.parse(url)

        print(
            f"{feed_type}: {len(feed.entries)} entries"
        )

        for entry in feed.entries:
            all_news.append({
                "company_name": entry.get("title"),
                "headline": entry.get("summary", ""),
                "feed_type": feed_type,
                "published": entry.get("published"),
                "url": entry.get("link"),
                "source": "NSE"
            })

    except Exception as e:
        print(f"{feed_type} ERROR:", e)

print("Total raw records:", len(all_news))

online_announcement: 0 entries
financial_result: 0 entries
board_meeting: 0 entries
corporate_action: 0 entries
annual_report: 0 entries
insider_trading: 0 entries
shareholding_pattern: 0 entries
corporate_governance: 0 entries
related_party_transaction: 0 entries
Total raw records: 0


In [114]:
nse_news = pd.DataFrame(all_news)

nse_news.head()

""


In [115]:
nse_news["feed_type"].value_counts()

KeyError: 'feed_type'

In [116]:
print(type(nse_feeds))
print(nse_feeds)

<class 'dict'>
{'online_announcement': 'https://nsearchives.nseindia.com/content/RSS/Online_announcements.xml', 'financial_result': 'https://nsearchives.nseindia.com/content/RSS/Financial_Results.xml', 'board_meeting': 'https://nsearchives.nseindia.com/content/RSS/Board_Meetings.xml', 'corporate_action': 'https://nsearchives.nseindia.com/content/RSS/Corporate_action.xml', 'annual_report': 'https://nsearchives.nseindia.com/content/RSS/Annual_Reports.xml', 'insider_trading': 'https://nsearchives.nseindia.com/content/RSS/InsiderTrading.xml', 'shareholding_pattern': 'https://nsearchives.nseindia.com/content/RSS/Shareholding_Pattern.xml', 'corporate_governance': 'https://nsearchives.nseindia.com/content/RSS/Corporate_Governance.xml', 'related_party_transaction': 'https://nsearchives.nseindia.com/content/RSS/Related_Party_Trans.xml'}


In [117]:
import feedparser

feed_test = {}

for feed_type, url in nse_feeds.items():
    feed = feedparser.parse(url)

    feed_test[feed_type] = len(feed.entries)

    print(
        f"{feed_type:<25} : {len(feed.entries)} entries | "
        f"bozo={feed.bozo}"
    )

online_announcement       : 0 entries | bozo=True
financial_result          : 0 entries | bozo=True
board_meeting             : 0 entries | bozo=True
corporate_action          : 0 entries | bozo=True
annual_report             : 0 entries | bozo=True
insider_trading           : 0 entries | bozo=True
shareholding_pattern      : 0 entries | bozo=True
corporate_governance      : 0 entries | bozo=True
related_party_transaction : 0 entries | bozo=True


In [118]:
pd.Series(feed_test).sort_values(ascending=False)

online_announcement          0
financial_result             0
board_meeting                0
corporate_action             0
annual_report                0
insider_trading              0
shareholding_pattern         0
corporate_governance         0
related_party_transaction    0
dtype: int64

In [119]:
import requests
import feedparser
import pandas as pd
import time

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "application/rss+xml, application/xml, "
        "text/xml, */*"
    ),
    "Referer": "https://www.nseindia.com/"
}


def fetch_nse_rss(url, feed_type):

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=30
        )

        response.raise_for_status()

        feed = feedparser.parse(response.content)

        news_data = []

        for entry in feed.entries:

            news_data.append({
                "company_name": entry.get("title"),
                "headline": entry.get("summary", ""),
                "published_at": entry.get("published"),
                "url": entry.get("link"),
                "source": "NSE",
                "feed_type": feed_type
            })

        df = pd.DataFrame(news_data)

        if not df.empty:

            df["published_at"] = pd.to_datetime(
                df["published_at"],
                format="%d-%b-%Y %H:%M:%S",
                errors="coerce"
            )

        return df

    except Exception as e:

        print(f"{feed_type}: ERROR -> {e}")

        return pd.DataFrame(
            columns=[
                "company_name",
                "headline",
                "published_at",
                "url",
                "source",
                "feed_type"
            ]
        )

In [120]:
all_nse_news = []

for feed_type, url in nse_feeds.items():

    print(f"\nFetching: {feed_type}")

    df = fetch_nse_rss(
        url,
        feed_type
    )

    print(f"Records: {len(df)}")

    if not df.empty:
        all_nse_news.append(df)

    time.sleep(1)


Fetching: online_announcement
Records: 91

Fetching: financial_result
Records: 3

Fetching: board_meeting
Records: 5

Fetching: corporate_action
Records: 77

Fetching: annual_report
Records: 20

Fetching: insider_trading
Records: 0

Fetching: shareholding_pattern
Records: 0

Fetching: corporate_governance
Records: 0

Fetching: related_party_transaction
Records: 20


In [121]:
nse_news = pd.concat(
    all_nse_news,
    ignore_index=True
)

print("\nTotal NSE records:", len(nse_news))


Total NSE records: 216


In [122]:
nse_news["feed_type"].value_counts()

feed_type
online_announcement          91
corporate_action             77
annual_report                20
related_party_transaction    20
board_meeting                 5
financial_result              3
Name: count, dtype: int64

In [123]:
print(
    "Unique companies:",
    nse_news["company_name"].nunique()
)

Unique companies: 193


In [124]:
nse_news["feed_type"].value_counts()

feed_type
online_announcement          91
corporate_action             77
annual_report                20
related_party_transaction    20
board_meeting                 5
financial_result              3
Name: count, dtype: int64

In [125]:
print("Total raw records:", len(nse_news))

Total raw records: 216


In [126]:
nse_news[
    [
        "company_name",
        "headline",
        "feed_type",
        "published_at"
    ]
].head(20)

,company_name,headline,feed_type,published_at
0,Neuland Laboratories Limited,Neuland Laboratories Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 20:00:11
1,GAIL (India) Limited,GAIL (India) Limited has informed the Exchange about change in Management |SUBJECT: Change in Management,online_announcement,2026-08-23 19:50:13
2,Vivo Collaboration Solutions Limited,"Vivo Collaboration Solutions Limited has submitted the Exchange a copy Srutinizers report of Annual General Meeting held on August 21, 2026. Furt...",online_announcement,2026-08-23 19:48:32
3,Standard Engineering Technology Limited,Standard Engineering Technology Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 19:36:14
4,Rajputana Biodiesel Limited,Rajputana Biodiesel Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 19:35:11
5,Aeron Composite Limited,Aeron Composite Limited has informed the Exchange about Board Meeting to be held on 29-Aug-2026 to consider Other business. |SUBJECT: Board Meetin...,online_announcement,2026-08-23 19:26:39
6,Aeron Composite Limited,Aeron Composite Limited has informed the Exchange about Resignation of Director/KMP/SMP |SUBJECT: Resignation of Director/KMP/SMP,online_announcement,2026-08-23 18:19:25
7,Aeron Composite Limited,Aeron Composite Limited has informed the Exchange regarding Resignation of Mr. Chirag Chandulal Patel as Managing Director and Director of the co...,online_announcement,2026-08-23 18:12:31
8,ABS Marine Services Limited,ABS Marine Services Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 18:03:31
9,APL Apollo Tubes Limited,APL Apollo Tubes Limited has informed the Exchange regarding Newspaper Advertisement of completion of despatch of the Notice of 41st Annual Genera...,online_announcement,2026-08-23 17:57:39


In [127]:
nse_news.groupby("feed_type")["company_name"].nunique()

feed_type
annual_report                19
board_meeting                 3
corporate_action             77
financial_result              2
online_announcement          85
related_party_transaction    14
Name: company_name, dtype: int64

In [128]:
nse_news.isna().sum()

company_name     0
headline         0
published_at    30
url              0
source           0
feed_type        0
dtype: int64

In [129]:
print("Total records:", len(nse_news))
print("Duplicate URLs:", nse_news["url"].duplicated().sum())
print("Duplicate headlines:", nse_news["headline"].duplicated().sum())

Total records: 216
Duplicate URLs: 132
Duplicate headlines: 39


In [130]:
import re

nse_news["headline_clean"] = (
    nse_news["headline"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
) 


In [131]:
nse_news["dedup_key"] = (
    nse_news["company_name"].astype(str).str.lower().str.strip()
    + "|"
    + nse_news["feed_type"].astype(str)
    + "|"
    + nse_news["headline_clean"]
)


In [132]:
print("Total records:", len(nse_news))
print("Duplicate composite records:", nse_news["dedup_key"].duplicated().sum())

Total records: 216
Duplicate composite records: 2


In [133]:
duplicates = nse_news[
    nse_news["dedup_key"].duplicated(keep=False)
].sort_values("dedup_key")

duplicates[
    [
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "url"
    ]
].head(30)

,company_name,headline,feed_type,published_at,url
191,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report,NaT,https://nsearchives.nseindia.com/annual_reports/SME_AR_30505_FONEBOX_2025_2026_U_11160032_21082026184437.pdf
193,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report,NaT,https://nsearchives.nseindia.com/annual_reports/SME_AR_30488_FONEBOX_2025_2026_A_11027498_21082026154321.pdf
14,Gujarat Themis Biosyn Limited,Gujarat Themis Biosyn Limited has informed the Exchange regarding the Amendment to AOA/MOA of the company. |SUBJECT: Amendment to AOA/MOA,online_announcement,2026-08-23 15:50:07,https://nsearchives.nseindia.com/corporate/GUJARATTHEMIS_23082026154955_Reg30AOAPBGTBL.pdf
24,Gujarat Themis Biosyn Limited,Gujarat Themis Biosyn Limited has informed the Exchange regarding the Amendment to AOA/MOA of the company. |SUBJECT: Amendment to AOA/MOA,online_announcement,2026-08-23 12:27:56,https://nsearchives.nseindia.com/corporate/GUJARATTHEMIS_23082026122743_Reg_30_AOA.pdf


In [134]:
nse_news[
    nse_news["company_name"].eq("Fonebox Retail Limited")
][
    [
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "url"
    ]
]

,company_name,headline,feed_type,published_at,url
191,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report,NaT,https://nsearchives.nseindia.com/annual_reports/SME_AR_30505_FONEBOX_2025_2026_U_11160032_21082026184437.pdf
193,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report,NaT,https://nsearchives.nseindia.com/annual_reports/SME_AR_30488_FONEBOX_2025_2026_A_11027498_21082026154321.pdf


In [135]:
nse_news[
    nse_news["dedup_key"].duplicated(keep=False)
][
    [
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "url"
    ]
]

,company_name,headline,feed_type,published_at,url
14,Gujarat Themis Biosyn Limited,Gujarat Themis Biosyn Limited has informed the Exchange regarding the Amendment to AOA/MOA of the company. |SUBJECT: Amendment to AOA/MOA,online_announcement,2026-08-23 15:50:07,https://nsearchives.nseindia.com/corporate/GUJARATTHEMIS_23082026154955_Reg30AOAPBGTBL.pdf
24,Gujarat Themis Biosyn Limited,Gujarat Themis Biosyn Limited has informed the Exchange regarding the Amendment to AOA/MOA of the company. |SUBJECT: Amendment to AOA/MOA,online_announcement,2026-08-23 12:27:56,https://nsearchives.nseindia.com/corporate/GUJARATTHEMIS_23082026122743_Reg_30_AOA.pdf
191,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report,NaT,https://nsearchives.nseindia.com/annual_reports/SME_AR_30505_FONEBOX_2025_2026_U_11160032_21082026184437.pdf
193,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report,NaT,https://nsearchives.nseindia.com/annual_reports/SME_AR_30488_FONEBOX_2025_2026_A_11027498_21082026154321.pdf


In [136]:
missing_pub = nse_news[
    nse_news["published_at"].isna()
]

missing_pub["feed_type"].value_counts()

feed_type
annual_report                20
related_party_transaction    10
Name: count, dtype: int64

In [137]:
missing_pub[
    [
        "company_name",
        "headline",
        "feed_type",
        "url"
    ]
].to_string(index=False)


'                                            company_name                      headline                 feed_type                                                                                                            url\n                             ABS Marine Services Limited        AS ON DATE : 22-AUG-26             annual_report  https://nsearchives.nseindia.com/annual_reports/SME_AR_30522_ABSMARINE_2025_2026_A_4942091_22082026195020.pdf\nGujarat Narmada Valley Fertilizers and Chemicals Limited        AS ON DATE : 22-AUG-26             annual_report          https://nsearchives.nseindia.com/annual_reports/AR_30521_GNFC_2025_2026_A_17228642_22082026180508.pdf\n                                 Sp Refractories Limited        AS ON DATE : 22-AUG-26             annual_report       https://nsearchives.nseindia.com/annual_reports/SME_AR_30520_SPRL_2025_2026_A_5465386_22082026173709.pdf\n                             Infinium Pharmachem Limited        AS ON DATE : 22-AUG-26             

In [138]:
missing_pub[
    ["feed_type", "headline"]
].head(50)

,feed_type,headline
176,annual_report,AS ON DATE : 22-AUG-26
177,annual_report,AS ON DATE : 22-AUG-26
178,annual_report,AS ON DATE : 22-AUG-26
179,annual_report,AS ON DATE : 22-AUG-26
180,annual_report,AS ON DATE : 22-AUG-26
181,annual_report,AS ON DATE : 22-AUG-26
182,annual_report,AS ON DATE : 22-AUG-26
183,annual_report,AS ON DATE : 22-AUG-26
184,annual_report,AS ON DATE : 22-AUG-26
185,annual_report,AS ON DATE : 22-AUG-26


In [139]:
import re

annual_mask = nse_news["feed_type"].eq("annual_report")

nse_news.loc[annual_mask, "event_date"] = (
    nse_news.loc[annual_mask, "headline"]
    .str.extract(
        r"AS ON DATE\s*:\s*(\d{2}-[A-Z]{3}-\d{2})",
        expand=False
    )
)

In [140]:
related_mask = nse_news["feed_type"].eq(
    "related_party_transaction"
)

nse_news.loc[related_mask, "event_date"] = (
    nse_news.loc[related_mask, "headline"]
    .str.extract(
        r"PERIOD END DATE\s*:\s*(\d{2}-[A-Z]{3}-\d{4})",
        expand=False
    )
)

In [141]:
nse_news["event_date"] = pd.to_datetime(
    nse_news["event_date"],
    errors="coerce"
)

/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_46573/1646266003.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  nse_news["event_date"] = pd.to_datetime(


In [142]:
nse_news[
    nse_news["event_date"].notna()
][
    [
        "company_name",
        "feed_type",
        "headline",
        "published_at",
        "event_date"
    ]
].tail(30)

,company_name,feed_type,headline,published_at,event_date
186,Xchanging Solutions Limited,annual_report,AS ON DATE : 21-AUG-26,NaT,2026-08-21
187,Vraj Iron and Steel Limited,annual_report,AS ON DATE : 21-AUG-26,NaT,2026-08-21
188,Balrampur Chini Mills Limited,annual_report,AS ON DATE : 21-AUG-26,NaT,2026-08-21
189,Shri Techtex Limited,annual_report,AS ON DATE : 21-AUG-26,NaT,2026-08-21
190,One 97 Communications Limited,annual_report,AS ON DATE : 21-AUG-26,NaT,2026-08-21
191,Fonebox Retail Limited,annual_report,AS ON DATE : 21-AUG-26,NaT,2026-08-21
192,Rex Pipes And Cables Industries Limited,annual_report,AS ON DATE : 21-AUG-26,NaT,2026-08-21
193,Fonebox Retail Limited,annual_report,AS ON DATE : 21-AUG-26,NaT,2026-08-21
194,S A Tech Software India Limited,annual_report,AS ON DATE : 21-AUG-26,NaT,2026-08-21
195,Pattech Fitwell Tube Components Limited,annual_report,AS ON DATE : 21-AUG-26,NaT,2026-08-21


In [143]:
print(
    nse_news["event_date"].notna().sum(),
    "records have event_date"
)

40 records have event_date


In [144]:
print(
    "Missing published_at:",
    nse_news["published_at"].isna().sum()
)

print(
    "Missing event_date:",
    nse_news["event_date"].isna().sum()
)

Missing published_at: 30
Missing event_date: 176


In [145]:
def get_date_status(row):
    if pd.notna(row["published_at"]) and pd.notna(row["event_date"]):
        return "published_and_event"
    elif pd.notna(row["published_at"]):
        return "published_only"
    elif pd.notna(row["event_date"]):
        return "event_only"
    else:
        return "no_date"

nse_news["date_status"] = nse_news.apply(
    get_date_status,
    axis=1
)

In [146]:
nse_news["date_status"].value_counts()

date_status
published_only         176
event_only              30
published_and_event     10
Name: count, dtype: int64

In [147]:
pd.crosstab(
    nse_news["feed_type"],
    nse_news["date_status"]
)

date_status,event_only,published_and_event,published_only
feed_type,,,
annual_report,20,0,0
board_meeting,0,0,5
corporate_action,0,0,77
financial_result,0,0,3
online_announcement,0,0,91
related_party_transaction,10,10,0


In [148]:
nse_news[
    nse_news["published_at"].isna()
][
    [
        "company_name",
        "feed_type",
        "headline",
        "event_date",
        "url"
    ]
].head(30)

,company_name,feed_type,headline,event_date,url
176,ABS Marine Services Limited,annual_report,AS ON DATE : 22-AUG-26,2026-08-22,https://nsearchives.nseindia.com/annual_reports/SME_AR_30522_ABSMARINE_2025_2026_A_4942091_22082026195020.pdf
177,Gujarat Narmada Valley Fertilizers and Chemicals Limited,annual_report,AS ON DATE : 22-AUG-26,2026-08-22,https://nsearchives.nseindia.com/annual_reports/AR_30521_GNFC_2025_2026_A_17228642_22082026180508.pdf
178,Sp Refractories Limited,annual_report,AS ON DATE : 22-AUG-26,2026-08-22,https://nsearchives.nseindia.com/annual_reports/SME_AR_30520_SPRL_2025_2026_A_5465386_22082026173709.pdf
179,Infinium Pharmachem Limited,annual_report,AS ON DATE : 22-AUG-26,2026-08-22,https://nsearchives.nseindia.com/annual_reports/SME_AR_30519_INFINIUM_2025_2026_A_2216015_22082026172907.pdf
180,Bannari Amman Sugars Limited,annual_report,AS ON DATE : 22-AUG-26,2026-08-22,https://nsearchives.nseindia.com/annual_reports/AR_30518_BANARISUG_2025_2026_A_59448263_22082026165031.pdf
181,Laxmi India Finance Limited,annual_report,AS ON DATE : 22-AUG-26,2026-08-22,https://nsearchives.nseindia.com/annual_reports/AR_30517_LAXMIINDIA_2025_2026_A_11673486_22082026155431.pdf
182,Astra Microwave Products Limited,annual_report,AS ON DATE : 22-AUG-26,2026-08-22,https://nsearchives.nseindia.com/annual_reports/AR_30516_ASTRAMICRO_2025_2026_A_8389931_22082026143020.pdf
183,Eastern Silk Industries Limited,annual_report,AS ON DATE : 22-AUG-26,2026-08-22,https://nsearchives.nseindia.com/annual_reports/AR_30515_EASTSILK_2025_2026_U_6588226_22082026142034.pdf
184,MIC Electronics Limited,annual_report,AS ON DATE : 22-AUG-26,2026-08-22,https://nsearchives.nseindia.com/annual_reports/AR_30514_MICEL_2025_2026_A_2371653_22082026125810.pdf
185,Shri Ahimsa Naturals Limited,annual_report,AS ON DATE : 22-AUG-26,2026-08-22,https://nsearchives.nseindia.com/annual_reports/SME_AR_30513_SHRIAHIMSA_2025_2026_A_5998714_22082026120521.pdf


In [149]:
before = len(nse_news)

nse_news = nse_news.drop_duplicates(
    subset=["url", "feed_type", "company_name"],
    keep="first"
).reset_index(drop=True)

after = len(nse_news)

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

Before: 216
After: 215
Removed: 1


In [150]:
print("Duplicate URLs:", nse_news["url"].duplicated().sum())

Duplicate URLs: 131


In [151]:
nse_news = nse_news[
    [
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "event_date",
        "date_status",
        "url",
        "source"
    ]
]

In [152]:
nse_news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   company_name  215 non-null    object        
 1   headline      215 non-null    object        
 2   feed_type     215 non-null    object        
 3   published_at  185 non-null    datetime64[ns]
 4   event_date    40 non-null     datetime64[ns]
 5   date_status   215 non-null    object        
 6   url           215 non-null    object        
 7   source        215 non-null    object        
dtypes: datetime64[ns](2), object(6)
memory usage: 13.6+ KB


In [153]:
nse_news.head()

,company_name,headline,feed_type,published_at,event_date,date_status,url,source
0,Neuland Laboratories Limited,Neuland Laboratories Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 20:00:11,NaT,published_only,https://nsearchives.nseindia.com/corporate/NEULANDLAB_23082026195217_SignedSEIntimationSpecialWindow.pdf,NSE
1,GAIL (India) Limited,GAIL (India) Limited has informed the Exchange about change in Management |SUBJECT: Change in Management,online_announcement,2026-08-23 19:50:13,NaT,published_only,https://nsearchives.nseindia.com/corporate/GAIL_23082026195007_Rupaji.pdf,NSE
2,Vivo Collaboration Solutions Limited,"Vivo Collaboration Solutions Limited has submitted the Exchange a copy Srutinizers report of Annual General Meeting held on August 21, 2026. Furt...",online_announcement,2026-08-23 19:48:32,NaT,published_only,https://nsearchives.nseindia.com/corporate/VIVO_23082026194641_RESULT_REPORT.pdf,NSE
3,Standard Engineering Technology Limited,Standard Engineering Technology Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 19:36:14,NaT,published_only,https://nsearchives.nseindia.com/corporate/SGLTPL_23082026193600_Intimation_Pre_AGM_Newspaper_Advt_23082026.pdf,NSE
4,Rajputana Biodiesel Limited,Rajputana Biodiesel Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 19:35:11,NaT,published_only,https://nsearchives.nseindia.com/corporate/RBPL_23082026193329_Rajputana_NP.pdf,NSE


In [154]:
import re

def normalize_company_name(name):
    name = str(name).upper().strip()

    # Remove common exchange suffixes
    name = re.sub(
        r"\s*[-–]\s*EX-DATE\s*:\s*\d{2}-[A-Z]{3}-\d{4}",
        "",
        name
    )

    # Normalize common company suffixes
    name = re.sub(r"\bLIMITED\b", "LTD", name)
    name = re.sub(r"\bPRIVATE LIMITED\b", "PVT LTD", name)

    # Remove punctuation
    name = re.sub(r"[^A-Z0-9& ]", " ", name)

    # Normalize spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name

In [158]:
companies_df["clean_company_name"] = (
    companies_df["db_company_name"]
    .apply(normalize_company_name)
)

In [159]:
nse_news["clean_company_name"] = (
    nse_news["company_name"]
    .apply(normalize_company_name)
)

In [160]:
print(
    nse_news[
        ["company_name", "clean_company_name"]
    ].head(20)
)

                                    company_name  \
0                   Neuland Laboratories Limited   
1                           GAIL (India) Limited   
2           Vivo Collaboration Solutions Limited   
3        Standard Engineering Technology Limited   
4                    Rajputana Biodiesel Limited   
5                        Aeron Composite Limited   
6                        Aeron Composite Limited   
7                        Aeron Composite Limited   
8                    ABS Marine Services Limited   
9                       APL Apollo Tubes Limited   
10                   Laxmi India Finance Limited   
11      Flexituff Ventures International Limited   
12  Knowledge Marine & Engineering Works Limited   
13                         Innova Captab Limited   
14                 Gujarat Themis Biosyn Limited   
15                  Manav Infra Projects Limited   
16            SBI Life Insurance Company Limited   
17                  Manav Infra Projects Limited   
18          

In [161]:
print(
    companies_df[
        ["db_company_name", "clean_company_name", "symbol"]
    ].head(20)
)

                             db_company_name  \
0                         20 Microns Limited   
1   21st Century Management Services Limited   
2                        360 ONE WAM LIMITED   
3                     3B Blackbio Dx Limited   
4                        3i Infotech Limited   
5                           3M India Limited   
6                   3P Land Holdings Limited   
7                     5Paisa Capital Limited   
8              63 moons technologies limited   
9              A2Z Infra Engineering Limited   
10                  AAA Technologies Limited   
11            Aadhar Housing Finance Limited   
12       Aakash Exploration Services Limited   
13     Aarey Drugs & Pharmaceuticals Limited   
14                   Aarnav Fashions Limited   
15                  Aaron Industries Limited   
16                  Aartech Solonics Limited   
17                       Aarti Drugs Limited   
18                  Aarti Industries Limited   
19                  Aarti Pharmalabs Lim

In [162]:
print(companies_df.columns.tolist())

['company_id', 'db_company_name', 'symbol', 'isin', 'normalized_company_name', 'clean_company_name']


In [163]:
# Create lookup from MySQL company master
company_lookup = (
    companies_df
    .drop_duplicates("clean_company_name")
    .set_index("clean_company_name")
)

In [164]:
nse_news["company_id"] = (
    nse_news["clean_company_name"]
    .map(company_lookup["company_id"])
)

nse_news["symbol"] = (
    nse_news["clean_company_name"]
    .map(company_lookup["symbol"])
)

nse_news["isin"] = (
    nse_news["clean_company_name"]
    .map(company_lookup["isin"])
)

In [165]:
matched = nse_news["company_id"].notna()

print("Total NSE records:", len(nse_news))
print("Matched records:", matched.sum())
print("Unmatched records:", (~matched).sum())
print(
    "Mapping success:",
    round(matched.mean() * 100, 2),
    "%"
)

Total NSE records: 215
Matched records: 151
Unmatched records: 64
Mapping success: 70.23 %


In [166]:
unmatched = (
    nse_news.loc[~matched]
    [
        [
            "company_name",
            "clean_company_name",
            "feed_type"
        ]
    ]
    .drop_duplicates()
    .sort_values("company_name")
)

print("Unmatched unique companies:", len(unmatched))

unmatched

Unmatched unique companies: 61


,company_name,clean_company_name,feed_type
43,Aditya Birla Sun Life BSE Sensex ETF,ADITYA BIRLA SUN LIFE BSE SENSEX ETF,online_announcement
42,Aditya Birla Sun Life CRISIL Liquid Overnight ETF,ADITYA BIRLA SUN LIFE CRISIL LIQUID OVERNIGHT ETF,online_announcement
32,Aditya Birla Sun Life Gold ETF - Growth,ADITYA BIRLA SUN LIFE GOLD ETF GROWTH,online_announcement
40,Aditya Birla Sun Life MF - Aditya Birla Sun Life BSE Top 10 Banks ETF,ADITYA BIRLA SUN LIFE MF ADITYA BIRLA SUN LIFE BSE TOP 10 BANKS ETF,online_announcement
41,Aditya Birla Sun Life MF- Aditya Birla Sun Life MSCI India ETF,ADITYA BIRLA SUN LIFE MF ADITYA BIRLA SUN LIFE MSCI INDIA ETF,online_announcement
...,...,...,...
88,SBI-ETF Gold,SBI ETF GOLD,online_announcement
81,SBI-ETF Nifty 50,SBI ETF NIFTY 50,online_announcement
83,SBI-ETF Nifty Bank,SBI ETF NIFTY BANK,online_announcement
87,SBI-ETF Nifty Next 50,SBI ETF NIFTY NEXT 50,online_announcement


In [167]:
nse_news[
    nse_news["company_id"].notna()
][
    [
        "company_id",
        "symbol",
        "isin",
        "company_name",
        "feed_type",
        "headline"
    ]
].head(20)

,company_id,symbol,isin,company_name,feed_type,headline
0,1548.0,NEULANDLAB,INE794A01010,Neuland Laboratories Limited,online_announcement,Neuland Laboratories Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication
1,762.0,GAIL,INE129A01019,GAIL (India) Limited,online_announcement,GAIL (India) Limited has informed the Exchange about change in Management |SUBJECT: Change in Management
2,3058.0,VIVO,INE0IA701014,Vivo Collaboration Solutions Limited,online_announcement,"Vivo Collaboration Solutions Limited has submitted the Exchange a copy Srutinizers report of Annual General Meeting held on August 21, 2026. Furt..."
3,1999.0,SETL,INE0M4D01010,Standard Engineering Technology Limited,online_announcement,Standard Engineering Technology Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication
4,2730.0,RAJPUTANA,INE0VHU01019,Rajputana Biodiesel Limited,online_announcement,Rajputana Biodiesel Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication
5,2776.0,AERON,INE0WL801011,Aeron Composite Limited,online_announcement,Aeron Composite Limited has informed the Exchange about Board Meeting to be held on 29-Aug-2026 to consider Other business. |SUBJECT: Board Meetin...
6,2776.0,AERON,INE0WL801011,Aeron Composite Limited,online_announcement,Aeron Composite Limited has informed the Exchange about Resignation of Director/KMP/SMP |SUBJECT: Resignation of Director/KMP/SMP
7,2776.0,AERON,INE0WL801011,Aeron Composite Limited,online_announcement,Aeron Composite Limited has informed the Exchange regarding Resignation of Mr. Chirag Chandulal Patel as Managing Director and Director of the co...
8,2837.0,ABSMARINE,INE0QRV01016,ABS Marine Services Limited,online_announcement,ABS Marine Services Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication
9,168.0,APLAPOLLO,INE702C01027,APL Apollo Tubes Limited,online_announcement,APL Apollo Tubes Limited has informed the Exchange regarding Newspaper Advertisement of completion of despatch of the Notice of 41st Annual Genera...


In [169]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 173.5 kB/s eta 0:00:00 kB/s eta 0:00:01

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [170]:
from rapidfuzz import process, fuzz

db_names = companies_df["clean_company_name"].dropna().unique()

unmatched_df = nse_news[
    nse_news["company_id"].isna()
].copy()

def fuzzy_match(name, choices, threshold=85):
    result = process.extractOne(
        name,
        choices,
        scorer=fuzz.token_set_ratio
    )
    
    if result is None:
        return None, 0
    
    match, score, _ = result
    
    if score >= threshold:
        return match, score
    
    return None, score


fuzzy_results = []

for name in unmatched_df["clean_company_name"].dropna().unique():
    match, score = fuzzy_match(name, db_names)
    
    fuzzy_results.append({
        "nse_name": name,
        "matched_db_name": match,
        "score": score
    })

fuzzy_df = pd.DataFrame(fuzzy_results)

fuzzy_df.sort_values("score", ascending=False).head(30)

,nse_name,matched_db_name,score
58,BALLARPUR INDUSTRIES LTD,NK INDUSTRIES LTD,90.322581
1,ADITYA BIRLA SUN LIFE NIFTY BANK ETF,None,84.000000
18,ADITYA BIRLA SUN LIFE MUTUAL FUND ABSL FTP SR TQ 1879 DAYS RP IDCW,None,84.000000
17,ADITYA BIRLA SUN LIFE MUTUAL FUND ABSL FTP SR TQ 1879 DAYS DP G,None,84.000000
16,ADITYA BIRLA SUN LIFE MUTUAL FUND ABSL FTP SR TQ 1879 DAYS RP G,None,84.000000
15,ADITYA BIRLA SUN LIFE MUTUAL FUND ADITYA BIRLA SUN LIFE SILVER ETF,None,84.000000
14,ADITYA BIRLA SUN LIFE MF ADITYA BIRLA SUN LIFE NIFTY HEALTHCARE ETF,None,84.000000
13,ADITYA BIRLA SUN LIFE BSE SENSEX ETF,None,84.000000
12,ADITYA BIRLA SUN LIFE CRISIL LIQUID OVERNIGHT ETF,None,84.000000
11,ADITYA BIRLA SUN LIFE MF ADITYA BIRLA SUN LIFE MSCI INDIA ETF,None,84.000000


In [171]:
# ETF / Mutual Fund type names ko temporarily exclude karo

exclude_keywords = [
    "ETF",
    "MUTUAL FUND",
    "MF ",
    "MF-",
    "FUND"
]

def is_fund_or_etf(name):
    name = str(name).upper()
    return any(keyword in name for keyword in exclude_keywords)


unmatched_normal = unmatched_df[
    ~unmatched_df["clean_company_name"].apply(is_fund_or_etf)
].copy()

print("Unmatched normal companies:", 
      unmatched_normal["clean_company_name"].nunique())

print("Unmatched ETF/MF:",
      unmatched_df[
          unmatched_df["clean_company_name"].apply(is_fund_or_etf)
      ]["clean_company_name"].nunique())

Unmatched normal companies: 4
Unmatched ETF/MF: 57


In [172]:
fuzzy_results = []

for name in unmatched_normal["clean_company_name"].dropna().unique():

    result = process.extractOne(
        name,
        db_names,
        scorer=fuzz.token_set_ratio
    )

    if result:
        match, score, _ = result

        fuzzy_results.append({
            "nse_name": name,
            "matched_db_name": match,
            "score": score
        })

fuzzy_df = pd.DataFrame(fuzzy_results)

fuzzy_df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

,nse_name,matched_db_name,score
0,BALLARPUR INDUSTRIES LTD,NK INDUSTRIES LTD,90.322581
1,BLUE BLENDS I LTD,BLUE STAR LTD,76.190476
2,SECUR CREDENTIALS LTD,T T LTD,75.000000
3,MINDSPACE BUSINESS PARKS REIT,FIVE STAR BUSINESS FINANCE LTD,57.627119


In [173]:
unmatched_normal[[
    "company_name",
    "clean_company_name",
    "feed_type",
    "headline",
    "url"
]]

,company_name,clean_company_name,feed_type,headline,url
92,Mindspace Business Parks REIT,MINDSPACE BUSINESS PARKS REIT,financial_result,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND AS/ NON IND A...,https://www.nseindia.com/companies-listing/corporate-filings-financial-results
195,Ballarpur Industries Limited,BALLARPUR INDUSTRIES LTD,related_party_transaction,PERIOD END DATE : 30-SEP-2024,https://nsearchives.nseindia.com/corporate/xbrl/RPT_1682435_18062026021817_WEB.xml
196,Ballarpur Industries Limited,BALLARPUR INDUSTRIES LTD,related_party_transaction,PERIOD END DATE : 31-MAR-2024,https://nsearchives.nseindia.com/corporate/xbrl/RPT_1682350_17062026072636_WEB.xml
197,Ballarpur Industries Limited,BALLARPUR INDUSTRIES LTD,related_party_transaction,PERIOD END DATE : 30-SEP-2023,https://nsearchives.nseindia.com/corporate/xbrl/RPT_1682008_16062026032832_WEB.xml
198,Ballarpur Industries Limited,BALLARPUR INDUSTRIES LTD,related_party_transaction,PERIOD END DATE : 31-MAR-2023,https://nsearchives.nseindia.com/corporate/xbrl/RPT_1682005_16062026032602_WEB.xml
199,SecUR Credentials Limited,SECUR CREDENTIALS LTD,related_party_transaction,PERIOD END DATE : 31-MAR-2024,https://nsearchives.nseindia.com/corporate/xbrl/RPT_1637801_20032026021302_WEB.xml
204,Blue Blends (I) Limited,BLUE BLENDS I LTD,related_party_transaction,PERIOD END DATE : 30-SEP-2024,https://nsearchives.nseindia.com/corporate/xbrl/RPT_1628010_19022026060602_WEB.xml


In [174]:
unmatched_normal["company_name"].tolist()

['Mindspace Business Parks REIT',
 'Ballarpur Industries Limited',
 'Ballarpur Industries Limited',
 'Ballarpur Industries Limited',
 'Ballarpur Industries Limited',
 'SecUR Credentials Limited',
 'Blue Blends (I) Limited']

In [175]:
unmatched_normal["url"].tolist()

['https://www.nseindia.com/companies-listing/corporate-filings-financial-results',
 'https://nsearchives.nseindia.com/corporate/xbrl/RPT_1682435_18062026021817_WEB.xml',
 'https://nsearchives.nseindia.com/corporate/xbrl/RPT_1682350_17062026072636_WEB.xml',
 'https://nsearchives.nseindia.com/corporate/xbrl/RPT_1682008_16062026032832_WEB.xml',
 'https://nsearchives.nseindia.com/corporate/xbrl/RPT_1682005_16062026032602_WEB.xml',
 'https://nsearchives.nseindia.com/corporate/xbrl/RPT_1637801_20032026021302_WEB.xml',
 'https://nsearchives.nseindia.com/corporate/xbrl/RPT_1628010_19022026060602_WEB.xml']

In [176]:
search_names = [
    "BALLARPUR",
    "BLUE BLENDS",
    "SECUR CREDENTIALS",
    "MINDSPACE"
]

for term in search_names:
    print("\n" + "="*60)
    print(term)

    result = companies_df[
        companies_df["db_company_name"]
        .str.upper()
        .str.contains(term, na=False)
    ]

    print(
        result[
            ["company_id", "db_company_name", "symbol", "isin"]
        ].to_string(index=False)
    )


BALLARPUR
Empty DataFrame
Columns: [company_id, db_company_name, symbol, isin]
Index: []

BLUE BLENDS
Empty DataFrame
Columns: [company_id, db_company_name, symbol, isin]
Index: []

SECUR CREDENTIALS
Empty DataFrame
Columns: [company_id, db_company_name, symbol, isin]
Index: []

MINDSPACE
Empty DataFrame
Columns: [company_id, db_company_name, symbol, isin]
Index: []


In [177]:
nse_news.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   company_name        215 non-null    object        
 1   headline            215 non-null    object        
 2   feed_type           215 non-null    object        
 3   published_at        185 non-null    datetime64[ns]
 4   event_date          40 non-null     datetime64[ns]
 5   date_status         215 non-null    object        
 6   url                 215 non-null    object        
 7   source              215 non-null    object        
 8   clean_company_name  215 non-null    object        
 9   company_id          151 non-null    float64       
 10  symbol              151 non-null    object        
 11  isin                151 non-null    object        
dtypes: datetime64[ns](2), float64(1), object(9)
memory usage: 20.3+ KB


In [178]:
nse_news.head()

,company_name,headline,feed_type,published_at,event_date,date_status,url,source,clean_company_name,company_id,symbol,isin
0,Neuland Laboratories Limited,Neuland Laboratories Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 20:00:11,NaT,published_only,https://nsearchives.nseindia.com/corporate/NEULANDLAB_23082026195217_SignedSEIntimationSpecialWindow.pdf,NSE,NEULAND LABORATORIES LTD,1548.0,NEULANDLAB,INE794A01010
1,GAIL (India) Limited,GAIL (India) Limited has informed the Exchange about change in Management |SUBJECT: Change in Management,online_announcement,2026-08-23 19:50:13,NaT,published_only,https://nsearchives.nseindia.com/corporate/GAIL_23082026195007_Rupaji.pdf,NSE,GAIL INDIA LTD,762.0,GAIL,INE129A01019
2,Vivo Collaboration Solutions Limited,"Vivo Collaboration Solutions Limited has submitted the Exchange a copy Srutinizers report of Annual General Meeting held on August 21, 2026. Furt...",online_announcement,2026-08-23 19:48:32,NaT,published_only,https://nsearchives.nseindia.com/corporate/VIVO_23082026194641_RESULT_REPORT.pdf,NSE,VIVO COLLABORATION SOLUTIONS LTD,3058.0,VIVO,INE0IA701014
3,Standard Engineering Technology Limited,Standard Engineering Technology Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 19:36:14,NaT,published_only,https://nsearchives.nseindia.com/corporate/SGLTPL_23082026193600_Intimation_Pre_AGM_Newspaper_Advt_23082026.pdf,NSE,STANDARD ENGINEERING TECHNOLOGY LTD,1999.0,SETL,INE0M4D01010
4,Rajputana Biodiesel Limited,Rajputana Biodiesel Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 19:35:11,NaT,published_only,https://nsearchives.nseindia.com/corporate/RBPL_23082026193329_Rajputana_NP.pdf,NSE,RAJPUTANA BIODIESEL LTD,2730.0,RAJPUTANA,INE0VHU01019


In [179]:
print("Total records:", len(nse_news))
print("Matched:", nse_news["company_id"].notna().sum())
print("Unmatched:", nse_news["company_id"].isna().sum())
print("Unique URLs:", nse_news["url"].nunique())

Total records: 215
Matched: 151
Unmatched: 64
Unique URLs: 84


In [180]:
nse_news["feed_type"].value_counts()

feed_type
online_announcement          91
corporate_action             77
annual_report                20
related_party_transaction    20
board_meeting                 5
financial_result              2
Name: count, dtype: int64

In [181]:
matched_news = nse_news[nse_news["company_id"].notna()].copy()

unmatched_news = nse_news[nse_news["company_id"].isna()].copy()

print("Matched records:", len(matched_news))
print("Unmatched records:", len(unmatched_news))

Matched records: 151
Unmatched records: 64


In [182]:
final_news = matched_news[
    [
        "company_id",
        "symbol",
        "isin",
        "company_name",
        "feed_type",
        "headline",
        "published_at",
        "event_date",
        "date_status",
        "url",
        "source"
    ]
].copy()

In [183]:
final_news.info()

<class 'pandas.core.frame.DataFrame'>
Index: 151 entries, 0 to 214
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   company_id    151 non-null    float64       
 1   symbol        151 non-null    object        
 2   isin          151 non-null    object        
 3   company_name  151 non-null    object        
 4   feed_type     151 non-null    object        
 5   headline      151 non-null    object        
 6   published_at  121 non-null    datetime64[ns]
 7   event_date    34 non-null     datetime64[ns]
 8   date_status   151 non-null    object        
 9   url           151 non-null    object        
 10  source        151 non-null    object        
dtypes: datetime64[ns](2), float64(1), object(8)
memory usage: 14.2+ KB


In [184]:
final_news.groupby(
    ["company_id", "symbol", "company_name"]
).size().sort_values(ascending=False).head(20)

company_id  symbol      company_name                                          
2776.0      AERON       Aeron Composite Limited                                   5
781.0       GAYAPROJ    Gayatri Projects Limited                                  4
2739.0      UHTL        United Heat Transfer Limited                              3
3105.0      MANAV       Manav Infra Projects Limited                              2
2888.0      FONEBOX     Fonebox Retail Limited                                    2
1293.0      LAXMIINDIA  Laxmi India Finance Limited                               2
885.0       GUJTHEM     Gujarat Themis Biosyn Limited                             2
2677.0      EEPL        Eppeltone Engineers Limited                               2
2696.0      SHRIAHIMSA  Shri Ahimsa Naturals Limited                              2
2837.0      ABSMARINE   ABS Marine Services Limited                               2
807.0       GLAND       Gland Pharma Limited                                     

In [185]:
final_news["feed_type"].value_counts()

feed_type
corporate_action             75
online_announcement          37
annual_report                20
related_party_transaction    14
board_meeting                 4
financial_result              1
Name: count, dtype: int64

In [186]:
final_news.groupby(
    "feed_type"
).size().sort_values(ascending=False)

feed_type
corporate_action             75
online_announcement          37
annual_report                20
related_party_transaction    14
board_meeting                 4
financial_result              1
dtype: int64

In [187]:
print("Published:")
print("Min:", final_news["published_at"].min())
print("Max:", final_news["published_at"].max())

print("\nEvent:")
print("Min:", final_news["event_date"].min())
print("Max:", final_news["event_date"].max())

Published:
Min: 2026-03-12 17:23:09
Max: 2026-08-23 20:00:11

Event:
Min: 2023-03-31 00:00:00
Max: 2026-08-22 00:00:00


In [188]:
print("========== FINAL DATA VALIDATION ==========")

# 1. Basic count
print("\nTotal records:", len(final_news))

# 2. Company mapping
print("\nCompany mapping:")
print("Mapped:", final_news["company_id"].notna().sum())
print("Unmapped:", final_news["company_id"].isna().sum())

# 3. Feed distribution
print("\nFeed distribution:")
print(final_news["feed_type"].value_counts())

# 4. Missing values
print("\nMissing values:")
print(final_news.isna().sum())

# 5. Duplicate URLs
print("\nDuplicate URLs:")
print(final_news["url"].duplicated().sum())

# 6. Duplicate composite records
print("\nDuplicate composite records:")

duplicate_mask = final_news.duplicated(
    subset=["company_id", "feed_type", "headline", "url"],
    keep=False
)

print(duplicate_mask.sum())

# 7. Date status
print("\nDate status:")
print(final_news["date_status"].value_counts())

# 8. Date ranges
print("\nPublished date range:")
print("Min:", final_news["published_at"].min())
print("Max:", final_news["published_at"].max())

print("\nEvent date range:")
print("Min:", final_news["event_date"].min())
print("Max:", final_news["event_date"].max())

========== FINAL DATA VALIDATION ==========

Total records: 151

Company mapping:
Mapped: 151
Unmapped: 0

Feed distribution:
feed_type
corporate_action             75
online_announcement          37
annual_report                20
related_party_transaction    14
board_meeting                 4
financial_result              1
Name: count, dtype: int64

Missing values:
company_id        0
symbol            0
isin              0
company_name      0
feed_type         0
headline          0
published_at     30
event_date      117
date_status       0
url               0
source            0
dtype: int64

Duplicate URLs:
76

Duplicate composite records:
0

Date status:
date_status
published_only         117
event_only              30
published_and_event      4
Name: count, dtype: int64

Published date range:
Min: 2026-03-12 17:23:09
Max: 2026-08-23 20:00:11

Event date range:
Min: 2023-03-31 00:00:00
Max: 2026-08-22 00:00:00


In [189]:
news_to_insert = final_news[
    [
        "company_id",
        "symbol",
        "isin",
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "event_date",
        "url",
        "source"
    ]
].copy()

print(news_to_insert.info())
print("\nTotal records:", len(news_to_insert))

<class 'pandas.core.frame.DataFrame'>
Index: 151 entries, 0 to 214
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   company_id    151 non-null    float64       
 1   symbol        151 non-null    object        
 2   isin          151 non-null    object        
 3   company_name  151 non-null    object        
 4   headline      151 non-null    object        
 5   feed_type     151 non-null    object        
 6   published_at  121 non-null    datetime64[ns]
 7   event_date    34 non-null     datetime64[ns]
 8   url           151 non-null    object        
 9   source        151 non-null    object        
dtypes: datetime64[ns](2), float64(1), object(7)
memory usage: 13.0+ KB
None

Total records: 151


In [190]:
print("Missing company_id:", news_to_insert["company_id"].isna().sum())
print("Missing symbol:", news_to_insert["symbol"].isna().sum())
print("Missing headline:", news_to_insert["headline"].isna().sum())
print("Missing feed_type:", news_to_insert["feed_type"].isna().sum())
print("Missing url:", news_to_insert["url"].isna().sum())

Missing company_id: 0
Missing symbol: 0
Missing headline: 0
Missing feed_type: 0
Missing url: 0


In [191]:
news_to_insert.head()

,company_id,symbol,isin,company_name,headline,feed_type,published_at,event_date,url,source
0,1548.0,NEULANDLAB,INE794A01010,Neuland Laboratories Limited,Neuland Laboratories Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 20:00:11,NaT,https://nsearchives.nseindia.com/corporate/NEULANDLAB_23082026195217_SignedSEIntimationSpecialWindow.pdf,NSE
1,762.0,GAIL,INE129A01019,GAIL (India) Limited,GAIL (India) Limited has informed the Exchange about change in Management |SUBJECT: Change in Management,online_announcement,2026-08-23 19:50:13,NaT,https://nsearchives.nseindia.com/corporate/GAIL_23082026195007_Rupaji.pdf,NSE
2,3058.0,VIVO,INE0IA701014,Vivo Collaboration Solutions Limited,"Vivo Collaboration Solutions Limited has submitted the Exchange a copy Srutinizers report of Annual General Meeting held on August 21, 2026. Furt...",online_announcement,2026-08-23 19:48:32,NaT,https://nsearchives.nseindia.com/corporate/VIVO_23082026194641_RESULT_REPORT.pdf,NSE
3,1999.0,SETL,INE0M4D01010,Standard Engineering Technology Limited,Standard Engineering Technology Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 19:36:14,NaT,https://nsearchives.nseindia.com/corporate/SGLTPL_23082026193600_Intimation_Pre_AGM_Newspaper_Advt_23082026.pdf,NSE
4,2730.0,RAJPUTANA,INE0VHU01019,Rajputana Biodiesel Limited,Rajputana Biodiesel Limited has informed the Exchange about Copy of Newspaper Publication |SUBJECT: Copy of Newspaper Publication,online_announcement,2026-08-23 19:35:11,NaT,https://nsearchives.nseindia.com/corporate/RBPL_23082026193329_Rajputana_NP.pdf,NSE
